In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [2]:
# function to get list of all race id in a season
# series key for now is 'series_1' for cup, 'series_2' for xfinity, 'series_3' for trucks
def get_race_ids(year, series_key):

    race_list_url = f"https://cf.nascar.com/cacher/{year}/race_list_basic.json"
    response = requests.get(race_list_url)
    data = response.json()
    races_df = pd.DataFrame(data[series_key])
    races_df_clean = races_df[['race_id', 'series_id', 'race_season', 'track_id', 'race_name', 'race_type_id', 'actual_laps']]
    return races_df_clean


In [3]:
# function to get track information 
def get_track_info():
    track_info_url = "https://cf.nascar.com/cacher/tracks.json"
    response = requests.get(track_info_url)
    data = response.json()
    df = pd.DataFrame(data['items'])
    df = df[['track_id', 'track_name', 'track_surface', 'track_type']]
    return df

### Getting race data for each race in 2025

In [29]:

year = 2025
series_key = 'series_1'
races = get_race_ids(year, series_key)
races = races[races['race_type_id'] ==1]
races.head()


,race_id,series_id,race_season,track_id,race_name,race_type_id,actual_laps
3,5546,1,2025,105,DAYTONA 500,1,201
4,5547,1,2025,111,Ambetter Health 400,1,266
5,5551,1,2025,214,EchoPark Automotive Grand Prix,1,95
6,5549,1,2025,84,Shriners Children's 500,1,312
7,5548,1,2025,42,Pennzoil 400 presented by Jiffy Lube,1,267


### track data for all tracks

In [30]:
track_df = get_track_info()
track_df.head()


,track_id,track_name,track_surface,track_type
0,4,Darlington Raceway,Asphalt,Intermediate
1,14,Bristol Motor Speedway,Concrete,Short Track
2,22,Martinsville Speedway,Asphalt,Short Track
3,26,Richmond Raceway,Asphalt,Short Track
4,38,Auto Club Speedway,Asphalt,Intermediate


### merging race data with the track data

In [31]:
races = races.merge(track_df, on='track_id', how='left')
races.head()

,race_id,series_id,race_season,track_id,race_name,race_type_id,actual_laps,track_name,track_surface,track_type
0,5546,1,2025,105,DAYTONA 500,1,201,Daytona International Speedway,Asphalt,Superspeedway
1,5547,1,2025,111,Ambetter Health 400,1,266,EchoPark Speedway,Asphalt,Drafting
2,5551,1,2025,214,EchoPark Automotive Grand Prix,1,95,Circuit of The Americas,Paved,Road Course
3,5549,1,2025,84,Shriners Children's 500,1,312,Phoenix Raceway,Asphalt,Intermediate
4,5548,1,2025,42,Pennzoil 400 presented by Jiffy Lube,1,267,Las Vegas Motor Speedway,Asphalt,Intermediate


In [35]:

def fetch_single_race(year,series_num, race_id):
    race_url = f"https://cf.nascar.com/cacher/{year}/{series_num}/{race_id}/lap-times.json"
    response = requests.get(race_url)
    data = response.json()
    df = pd.json_normalize(
        data['laps'],
        record_path='Laps',
        meta=['Number', 'FullName', 'NASCARDriverID']
    )

    flag_key = {
    0: 'None',
    1: 'Green',
    2: 'Yellow',
    3: 'Red',
    4: 'White',
    5: 'Checkered',
    8: 'Hot Track',
    9: 'Cold Track'}
    
    flag_status = pd.DataFrame(data['flags'])
    flag_status["FlagName"] = flag_status["FlagState"].map(flag_key).fillna("Unknown")
    df = pd.merge(df, flag_status, left_on = 'Lap', right_on = 'LapsCompleted', how = 'left')

    return df


def get_lap_data(year, series_num, race_ids):
    lap_data = []
    for race_id in race_ids: 
        race_data = fetch_single_race(year, series_num,race_id)
        race_data['race_id'] = race_id
        lap_data.append(race_data)

    lap_df = pd.concat(lap_data, ignore_index=True)
    return lap_df



race_ids = races['race_id'].tolist()
series_num = 1

lap_df = get_lap_data(year, series_num, race_ids)
lap_df.head()

,Lap,LapTime,LapSpeed,RunningPos,Number,FullName,NASCARDriverID,LapsCompleted,FlagState,FlagName,race_id
0,0,NaN,NaN,5,24,William Byron,4184,0.0,8.0,Hot Track,5546
1,1,52.135,172.629,5,24,William Byron,4184,1.0,1.0,Green,5546
2,2,47.861,188.045,5,24,William Byron,4184,2.0,1.0,Green,5546
3,3,47.966,187.633,4,24,William Byron,4184,3.0,1.0,Green,5546
4,4,47.957,187.668,2,24,William Byron,4184,4.0,1.0,Green,5546
